# Employee Attrition Prediction System - Full ML Lifecycle

## Phase 1 - Problem Understanding
**Background:** Employee attrition is the gradual reduction in staff numbers that occurs as employees retire or resign. It is a major cost to organizations.
**Business Problem:** High attrition rates lead to increased hiring costs, lost productivity, and lowered morale.
**Problem Statement:** Predict which employees are likely to leave the organization so that HR can take proactive retention measures.
**Objectives:** 
- Analyze employee data to find key factors driving attrition.
- Build predictive models to classify employee attrition.
**Scope:** The scope is limited to the provided HR dataset.
**Expected Outcome:** A robust classification model identifying at-risk employees and the factors contributing to their risk.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import joblib

warnings.filterwarnings('ignore')
%matplotlib inline

# Ensure Model directory exists
os.makedirs('../Model', exist_ok=True)

## Phase 2 - Data Collection

In [ ]:
dataset_path = '../Dataset/WA_Fn-UseC_-HR-Employee-Attrition.csv'
try:
    df = pd.read_csv(dataset_path)
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print(f"Error: Dataset not found at {dataset_path}. Please run download_dataset.py first.")
    
display(df.head())

In [ ]:
print("Dataset Information:")
df.info()
print("\nDataset Shape:", df.shape)
print("\nColumn Descriptions (Data Types):\n", df.dtypes)
print("\nStatistical Summary:")
display(df.describe())

## Phase 3 - Data Preprocessing

In [ ]:
# Handle missing values
print("Missing values:\n", df.isnull().sum().sum())

# Remove duplicate records
df.drop_duplicates(inplace=True)
print("Shape after dropping duplicates:", df.shape)

# Drop redundant/useless columns
columns_to_drop = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')

# Detect and treat outliers (using IQR for MonthlyIncome as an example)
Q1 = df['MonthlyIncome'].quantile(0.25)
Q3 = df['MonthlyIncome'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['MonthlyIncome'] < lower_bound) | (df['MonthlyIncome'] > upper_bound)]
print(f"Number of outliers in MonthlyIncome: {len(outliers)}")
# We will cap outliers rather than dropping to preserve data
df['MonthlyIncome'] = np.clip(df['MonthlyIncome'], lower_bound, upper_bound)

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Encode categorical variables
label_encoders = {}
categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Save encoders for deployment
joblib.dump(label_encoders, '../Model/encoder.pkl')
print("Encoders saved to ../Model/encoder.pkl")

## Phase 4 - Exploratory Data Analysis (EDA)

In [ ]:
# Univariate Analysis
plt.figure(figsize=(6,4))
sns.countplot(x='Attrition', data=df)
plt.title('Target Variable Distribution (Attrition)')
plt.show()
print("Insight: The dataset is highly imbalanced with 'No' (0) being much higher than 'Yes' (1).")

In [ ]:
# Bivariate Analysis
plt.figure(figsize=(10,6))
sns.boxplot(x='Attrition', y='MonthlyIncome', data=df)
plt.title('Monthly Income vs Attrition')
plt.show()
print("Insight: Employees who leave tend to have a slightly lower median Monthly Income.")

In [ ]:
# Multivariate Analysis & Correlation
plt.figure(figsize=(20, 15))
corr = df.corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()
print("Insight: JobLevel and MonthlyIncome are highly correlated. TotalWorkingYears is also highly correlated with JobLevel.")

## Data Balancing (SMOTE) & Splitting

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

X = df.drop('Attrition', axis=1)
y = df['Attrition']

# Train-test split (80:20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Apply SMOTE to training data only to avoid data leakage
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Original training target distribution:\n{y_train.value_counts()}")
print(f"SMOTE training target distribution:\n{y_train_smote.value_counts()}")

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test)

# Save scaler for deployment
joblib.dump(scaler, '../Model/scaler.pkl')
print("Scaler saved to ../Model/scaler.pkl")

## Phase 5 - Feature Engineering & Selection

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Feature Selection using Random Forest Feature Importance
rf_fs = RandomForestClassifier(random_state=42)
rf_fs.fit(X_train_scaled, y_train_smote)

importances = rf_fs.feature_importances_
feature_names = X.columns
feature_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=feature_imp_df.head(15))
plt.title('Top 15 Features by Random Forest Importance')
plt.show()
print("Insight: MonthlyIncome, Age, TotalWorkingYears, and DistanceFromHome are among the most important features.")

## Phase 6 - Model Building & Phase 7 - Model Evaluation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import time

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'),
    'SVM': SVC(random_state=42, probability=True),
    'KNN': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB(),
    'MLP Neural Network': MLPClassifier(random_state=42, max_iter=1000)
}

results = []
trained_models = {}

for name, model in models.items():
    start_time = time.time()
    
    # Train model
    model.fit(X_train_scaled, y_train_smote)
    trained_models[name] = model
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else [0]*len(y_test)
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob) if hasattr(model, 'predict_proba') else 0
    
    end_time = time.time()
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'ROC-AUC': roc_auc,
        'Time (s)': round(end_time - start_time, 2)
    })

results_df = pd.DataFrame(results).sort_values(by='F1 Score', ascending=False)
display(results_df)

In [ ]:
# Identify and save the best model
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]

print(f"\nBest Model based on F1 Score: {best_model_name}")

joblib.dump(best_model, '../Model/best_model.pkl')
print(f"Best model '{best_model_name}' saved to ../Model/best_model.pkl")

# Detailed evaluation of best model
y_pred_best = best_model.predict(X_test_scaled)
print("\nClassification Report for Best Model:")
print(classification_report(y_test, y_pred_best))

plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test, y_pred_best), annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix: {best_model_name}')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()